# NRW-Pipeline — Basisdatensatz
**Strom: CO2online Stromspiegel 2025 (ohne elektrische Warmwasserbereitung)**

Dieses Notebook baut die gemeinsame Arbeitsdatei `nrw_buildings.parquet` aus dem
rohen OSM-Gebaeudedatensatz auf. Es berechnet Strombedarf, Waermebedarf und
PV-Potenzial fuer alle ~4,1 Mio. NRW-Wohngebaeude.

Das Notebook ist **optional** — es ist nur noetig, wenn die Arbeitsdatei von
Grund auf neu aufgebaut werden soll (`DEA_combined.parquet` als Rohquelle).

| Abschnitt | Aufgabe |
|---|---|
| 1 | Imports und Konfiguration |
| 2 | CO2online-Lookup-Tabelle |
| 3 | Zensus-2022-TABULA-Statistiken |
| 4 | Rohdaten laden |
| 5 | Grundflaeche |
| 6 | Wohnflaeche (EnEV 2009) |
| 7 | Strombedarf |
| 8 | PV-Ertrag ueber die PVGIS-API |
| 9 | Zonen-Zuordnung |
| 10 | Waermebedarf (TABULA) |
| 11 | Parquet speichern |

**Hinweis:** Die Abschnitte 1-11 benoetigen auf dem vollen 4,1-Mio-Datensatz
etwa 10-15 Minuten Laufzeit.

## 1. Konfiguration

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from pyproj import Transformer

# Relative Pfade — funktioniert auf jedem Rechner unabhaengig vom Benutzernamen.
# Im NL-Pendant (netherlands-building-clustering-main) liegt die gemeinsame
# Arbeitsdatei ebenfalls auf dieser Ebene: nl_buildings.parquet.
BASE_DIR     = "."
INPUT_FILE   = os.path.join(BASE_DIR, "DEA_combined.parquet")
HEAT_FILE    = os.path.join(BASE_DIR, "final_tabula.xlsx")
TRY_FILE     = os.path.join(BASE_DIR, "TRY_polygons.geojson")
OUTPUT_FILE  = os.path.join(BASE_DIR, "nrw_buildings.parquet")

print("Imports OK")
print(f"Eingabe : {INPUT_FILE}")
print(f"Waerme  : {HEAT_FILE}")
print(f"Zonen   : {TRY_FILE}")
print(f"Ausgabe : {OUTPUT_FILE}")

Imports OK
Eingabe : ./DEA_combined.parquet
Waerme  : ./final_tabula.xlsx
Zonen   : ./TRY_polygons.geojson
Ausgabe : ./nrw_buildings.parquet


In [2]:
# Konstanten zur Gebaeudegeometrie
ROOF_CORRECTION   = 3      # Meter, die von der Gebaeudehoehe fuer das Dach abgezogen werden
MIN_HEATED_HEIGHT = 3.5    # minimale beheizte Hoehe in Metern

# PV-Anlagenparameter
ROOF_FRACTION    = 0.20    # Anteil der Grundflaeche, der fuer PV genutzt wird
PANEL_EFFICIENCY = 0.20    # 20 % Modulwirkungsgrad (kristallines Silizium)
TILT             = 35      # Modulneigung in Grad
AZIMUTH          = -45     # Modulazimut (-45 = Suedwest, 0 = Sued)
SYSTEM_LOSS      = 14       # Systemverluste in Prozent (PVGIS-Standard)

# Zwei PVGIS-Referenzstandorte, die NRW abdecken
ESSEN          = {"name": "Essen",          "lat": 51.455, "lon": 7.011}
BAD_MARIENBERG = {"name": "Bad Marienberg", "lat": 50.651, "lon": 7.952}

SIZE_CLASSES = ["SFH", "TH", "MFH", "AB"]
print("Konfiguration geladen")

Konfiguration geladen


## 2. CO2online-Lookup-Tabelle

Strombedarf je Wohnung und Jahr nach Personenzahl und Gebaeudekategorie.

**Quelle:** CO2online Stromspiegel via Statista (Mai 2025)
**Werte:** OHNE elektrische Warmwasserbereitung

**Warum ohne elektrische Warmwasserbereitung:**
Das SESMG optimiert die Warmwasserbereitung separat. Eine Einbeziehung hier
wuerde den Strombedarf in der Simulation doppelt zaehlen.

**Hinweis zu den 4-Personen-Werten:** Stand April 2023 — aktuellste verfuegbare
Zahl auf Statista zum Zeitpunkt der Analyse.

**Kategorieregel:**
- SFH und TH → Spalte EFH (Ein-Wohnung-Gebaeude)
- MFH und AB → Spalte MFH (Wohnungen in Mehrfamilienhaeusern, kleinerer Bedarf je Wohnung)

In [3]:
# CO2online Stromspiegel 2025 — ohne elektrische Warmwasserbereitung
# Schluessel: Personen (1-5), Wert: (EFH kWh/Jahr, MFH/AB kWh/Jahr)
ELEC_BY_PERSONS = {
    1: (1_800, 1_200),   # Statista Mai 2025
    2: (2_700, 1_900),   # Statista Mai 2025
    3: (3_500, 2_400),   # Statista Mai 2025
    4: (4_000, 2_900),   # Statista April 2023 (aktuellster Stand)
    5: (4_500, 3_100),   # Statista Mai 2025
}

# Tabelle zur Referenz ausgeben
print("CO2online Stromspiegel 2025 — ohne elektrische Warmwasserbereitung")
print(f'  {"Personen":<10} {"SFH/TH (kWh/Jahr)":<20} {"MFH/AB (kWh/Jahr)"}')
print("-" * 50)
for p, (efh, mfh) in ELEC_BY_PERSONS.items():
    print(f"  {p:<10} {efh:<20,} {mfh:,}")

CO2online Stromspiegel 2025 — ohne elektrische Warmwasserbereitung
  Personen   SFH/TH (kWh/Jahr)    MFH/AB (kWh/Jahr)
--------------------------------------------------
  1          1,800                1,200
  2          2,700                1,900
  3          3,500                2,400
  4          4,000                2,900
  5          4,500                3,100


## 3. Zensus-2022-TABULA-Statistiken

Mittlere Wohnungsgroesse und mittlere Bewohnerzahl je TABULA-Typ.

**Quelle:** Zensus 2022 (Destatis), gefiltert auf homogene 100-m-Rasterzellen,
in denen nur ein Gebaeudetyp existiert — stellt sicher, dass die Mittelwerte
nicht durch gemischte Quartiere verfaelscht werden.

Diese beiden Werte werden in Abschnitt 7 benoetigt fuer:
- Zahl der Wohnungen je Gebaeude (Al / mittlere Wohnungsgroesse)
- Zahl der Personen je Wohnung (mittlere Bewohnerzahl)

In [4]:
TABULA_STATS = pd.DataFrame([
    # Appartmentbloecke (AB)
    ("DE.N.AB.02",  63, 1), ("DE.N.AB.03",  88, 2), ("DE.N.AB.04",  69, 2),
    ("DE.N.AB.05",  69, 2), ("DE.N.AB.06",  69, 2), ("DE.N.AB.07",  70, 2),
    ("DE.N.AB.08",  64, 2), ("DE.N.AB.09",  69, 2), ("DE.N.AB.10",  68, 2),
    ("DE.N.AB.11",  58, 1), ("DE.N.AB.12",  62, 2),
    # Mehrfamilienhaeuser (MFH)
    ("DE.N.MFH.01", 80, 2), ("DE.N.MFH.02", 82, 2), ("DE.N.MFH.03", 73, 2),
    ("DE.N.MFH.04", 74, 2), ("DE.N.MFH.05", 74, 2), ("DE.N.MFH.06", 74, 2),
    ("DE.N.MFH.07", 79, 2), ("DE.N.MFH.08", 78, 2), ("DE.N.MFH.09", 79, 2),
    ("DE.N.MFH.10", 84, 2), ("DE.N.MFH.11", 76, 2), ("DE.N.MFH.12", 77, 2),
    # Einfamilienhaeuser (SFH)
    ("DE.N.SFH.01", 118, 2), ("DE.N.SFH.02", 119, 2), ("DE.N.SFH.03", 116, 2),
    ("DE.N.SFH.04", 123, 2), ("DE.N.SFH.05", 123, 2), ("DE.N.SFH.06", 123, 2),
    ("DE.N.SFH.07", 128, 2), ("DE.N.SFH.08", 127, 2), ("DE.N.SFH.09", 129, 2),
    ("DE.N.SFH.10", 135, 3), ("DE.N.SFH.11", 142, 3), ("DE.N.SFH.12", 141, 3),
    # Reihenhaeuser (TH)
    ("DE.N.TH.01",   99, 2), ("DE.N.TH.02", 100, 2), ("DE.N.TH.03",  99, 2),
    ("DE.N.TH.04",  107, 2), ("DE.N.TH.05", 107, 2), ("DE.N.TH.06", 107, 2),
    ("DE.N.TH.07",  117, 2), ("DE.N.TH.08", 117, 2), ("DE.N.TH.09", 121, 3),
    ("DE.N.TH.10",  124, 3), ("DE.N.TH.11", 116, 3), ("DE.N.TH.12", 114, 3),
], columns=["tabula_key", "avg_dwelling_size_m2", "avg_inhabitants"])

# Ersatzwerte fuer TABULA-Typen, die nicht in der Tabelle stehen
FALLBACK_SIZE = {"SFH": 120, "TH": 108, "MFH": 74, "AB": 69}
FALLBACK_INH  = {"SFH": 2,   "TH": 2,   "MFH": 2,  "AB": 2}

print(f"TABULA_STATS geladen: {len(TABULA_STATS)} TABULA-Typen")
print(TABULA_STATS.head(5).to_string(index=False))

TABULA_STATS geladen: 47 TABULA-Typen
tabula_key  avg_dwelling_size_m2  avg_inhabitants
DE.N.AB.02                    63                1
DE.N.AB.03                    88                2
DE.N.AB.04                    69                2
DE.N.AB.05                    69                2
DE.N.AB.06                    69                2


## 4. Rohdaten laden

Die Rohquelle ist `DEA_combined.parquet` — der OSM-Gebaeudedatensatz fuer NRW.
Er enthaelt Grundflaechengeometrie, Hoehe, TABULA-Typ, Sanierungszustand und
Baujahr.

Das Koordinatenreferenzsystem wird automatisch aus den Geometrie-Koordinaten
erkannt.

In [5]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Nicht gefunden: {INPUT_FILE}\n"
                             "DEA_combined.parquet muss im Projektstamm liegen.")

gdf = gpd.read_parquet(INPUT_FILE)
print(f"Gebaeude          : {len(gdf):,}")
print(f"Geometrie-Spalte  : {gdf.geometry.name}")
print(f"Spalten           : {gdf.columns.tolist()}")

gdf.set_crs("EPSG:3035", allow_override=True, inplace=True)
gdf.to_crs("EPSG:25832", inplace=True)

print(f'tabula_type Bsp.  : {gdf["tabula_type"].iloc[0]}')

Gebaeude          : 4,133,323
Geometrie-Spalte  : footprint
Spalten           : ['footprint', 'height_m', 'id', 'construction_year', 'size_class', 'refurbishment_state', 'tabula_type']
tabula_type Bsp.  : DE.N.TH.05.Gen.ReEx.001.003


## 5. Grundflaeche

Wird direkt aus der Gebaeudepolygon-Geometrie berechnet. Das ist die
Grundrissflaeche des Gebaeudes in Quadratmetern.

In [6]:
gdf["area_m2"] = gdf.geometry.area
print("Grundflaeche:")
print(f'  min  : {gdf["area_m2"].min():.1f} m2')
print(f'  mittel: {gdf["area_m2"].mean():.1f} m2')
print(f'  max  : {gdf["area_m2"].max():.1f} m2')

Grundflaeche:
  min  : 1.0 m2
  mittel: 123.5 m2
  max  : 20946.2 m2


## 6. Wohnflaeche (EnEV 2009 / Loga et al. 2012)

Die konditionierte Wohnflaeche Al wird aus der Gebaeudegeometrie abgeleitet:

```
heated_height = max(h_building - 3.0,  3.5)   # Dach abziehen, Minimum 3,5 m
volume_m3     = area_m2 × heated_height
Au_m2         = 0.32 × volume_m3               # EnEV 2009
Al_m2         = Au_m2 / 1.3                    # Loga et al. 2012
```

`ROOF_CORRECTION = 3,0 m` zieht den nicht beheizten Dachraum von der
Gebaeudehoehe ab. `MIN_HEATED_HEIGHT = 3,5 m` verhindert unrealistisch kleine
Werte bei sehr flachen Gebaeuden.

Dieses Al wird in Abschnitt 7 verwendet.

In [7]:
gdf["heated_height"] = (gdf["height_m"] - ROOF_CORRECTION).clip(lower=MIN_HEATED_HEIGHT)
gdf["volume_m3"]     = gdf["area_m2"] * gdf["heated_height"]
gdf["Au_m2"]         = 0.32 * gdf["volume_m3"]
gdf["Al_m2"]         = gdf["Au_m2"] / 1.3

print("Wohnflaeche (Al):")
print(f'  min  : {gdf["Al_m2"].min():.1f} m2')
print(f'  mittel: {gdf["Al_m2"].mean():.1f} m2')
print(f'  max  : {gdf["Al_m2"].max():.1f} m2')

Wohnflaeche (Al):
  min  : 0.9 m2
  mittel: 220.4 m2
  max  : 51269.1 m2


## 7. Strombedarf

Vier Teilschritte:

**7.1 — TABULA-Schluessel extrahieren**
Extrahiert den vierstelligen TABULA-Schluessel aus dem vollstaendigen
tabula_type-String. Beispiel: `DE.N.TH.05.Gen.ReEx.001.003` → `DE.N.TH.05`

**7.2 — Zensus-2022-Statistiken mergen**
Verknuepft mittlere Wohnungsgroesse und mittlere Bewohnerzahl je TABULA-Typ.

**7.3 — Zahl der Wohnungen**
```
n_dwellings = max(1, round(Al / avg_dwelling_size))
```
Minimum 1 — jedes Gebaeude hat mindestens eine Wohnung.

**7.4 — Strom-Lookup**
```
persons  = max(1, min(5, round(avg_inhabitants)))
per_dw   = CO2online[persons][EFH oder MFH]
elec_kwh = n_dwellings × per_dw
```

In [8]:
# 7.1 Vierstelligen TABULA-Schluessel extrahieren
gdf["tabula_key"] = gdf["tabula_type"].str.split(".").str[:4].str.join(".")
print(f'tabula_key-Beispiele: {gdf["tabula_key"].unique()[:4].tolist()}')

tabula_key-Beispiele: ['DE.N.TH.05', 'DE.N.TH.04', 'DE.N.SFH.05', 'DE.N.MFH.06']


In [9]:
# 7.2 Zensus-2022-Statistiken mergen
gdf = gdf.merge(TABULA_STATS, on="tabula_key", how="left")

missing = gdf["avg_dwelling_size_m2"].isna()
gdf.loc[missing, "avg_dwelling_size_m2"] = gdf.loc[missing, "size_class"].map(FALLBACK_SIZE)
gdf.loc[missing, "avg_inhabitants"]      = gdf.loc[missing, "size_class"].map(FALLBACK_INH)

print(f"Ueber TABULA_STATS zugeordnet : {(~missing).sum():,} Gebaeude")
print(f"Ersatzwerte verwendet          : {missing.sum():,} Gebaeude")

Ueber TABULA_STATS zugeordnet : 4,133,323 Gebaeude
Ersatzwerte verwendet          : 0 Gebaeude


In [10]:
# 7.3 Zahl der Wohnungen je Gebaeude
gdf["n_dwellings"] = (
    gdf["Al_m2"] / gdf["avg_dwelling_size_m2"]
).round().clip(lower=1).astype(int)

# SFH und TH haben definitionsgemaess immer genau 1 Wohnung
is_efh = gdf["size_class"].isin(["SFH", "TH"])
gdf.loc[is_efh, "n_dwellings"] = 1

print("Verteilung der Wohnungszahl (alle Gebaeude):")
print(gdf["n_dwellings"].value_counts().sort_index().head(10).to_string())
print()
print("SFH-Wohnungszahl (muss durchgaengig 1 sein):")
print(gdf[gdf["size_class"]=="SFH"]["n_dwellings"].value_counts().to_dict())
print("TH-Wohnungszahl (muss durchgaengig 1 sein):")
print(gdf[gdf["size_class"]=="TH"]["n_dwellings"].value_counts().to_dict())

Verteilung der Wohnungszahl (alle Gebaeude):
n_dwellings
1     3325783
2      122981
3      125297
4      119177
5      104939
6       89049
7       66135
8       45065
9       30963
10      21582

SFH-Wohnungszahl (muss durchgaengig 1 sein):
{1: 2723098}
TH-Wohnungszahl (muss durchgaengig 1 sein):
{1: 507633}


In [11]:
# 7.4 CO2online-Lookup — vektorisierte NumPy-Indizierung (deutlich schneller als apply)
persons = gdf["avg_inhabitants"].round().clip(lower=1, upper=5).astype(int)

EFH_KWH = np.array([0, 1_800, 2_700, 3_500, 4_000, 4_500])  # Index = Personen
MFH_KWH = np.array([0, 1_200, 1_900, 2_400, 2_900, 3_100])

is_efh = gdf["size_class"].isin(["SFH", "TH"])
gdf["demand_per_dw_kwh"] = np.where(is_efh, EFH_KWH[persons], MFH_KWH[persons])
gdf["elec_kwh"]      = gdf["n_dwellings"] * gdf["demand_per_dw_kwh"]

print("Strombedarf:")
print(f'  min    : {gdf["elec_kwh"].min():>12,.0f} kWh/Jahr')
print(f'  mittel : {gdf["elec_kwh"].mean():>12,.0f} kWh/Jahr')
print(f'  max    : {gdf["elec_kwh"].max():>12,.0f} kWh/Jahr')
print(f'  gesamt : {gdf["elec_kwh"].sum()/1e9:.3f} TWh/Jahr')
print(f'  BDEW-Referenz: 31.96 TWh/Jahr')
print(f'  Abweichung    : {(gdf["elec_kwh"].sum()/1e9 - 31.96)/31.96*100:.1f}%')
print(f"\nNach Groessenklasse:")
for sc in SIZE_CLASSES:
    sub = gdf[gdf["size_class"] == sc]
    print(f'  {sc}: mittel={sub["elec_kwh"].mean():>10,.0f}  median={sub["elec_kwh"].median():>10,.0f}  n={len(sub):>9,}')

Strombedarf:
  min    :        1,200 kWh/Jahr
  mittel :        4,483 kWh/Jahr
  max    :    1,316,700 kWh/Jahr
  gesamt : 18.530 TWh/Jahr
  BDEW-Referenz: 31.96 TWh/Jahr
  Abweichung    : -42.0%

Nach Groessenklasse:
  SFH: mittel=     2,728  median=     2,700  n=2,723,098
  TH: mittel=     2,786  median=     2,700  n=  507,633
  MFH: mittel=    10,504  median=     7,600  n=  897,247
  AB: mittel=    49,102  median=    34,200  n=    5,345


## 8. PV-Ertrag ueber die PVGIS-API

Ruft den jaehrlichen PV-Ertrag in kWh je kWp fuer zwei NRW-Referenzstandorte ab.
PVGIS (Photovoltaic Geographical Information System) ist die offizielle
Sonnenstrahlungs-Datenbank der EU.

Zwei Standorte decken die Spannweite der Einstrahlung in NRW ab:
- **Essen** (Nordwest-NRW) — geringere Einstrahlung
- **Bad Marienberg** (Suedost-NRW) — hoehere Einstrahlung

Schlaegt der API-Aufruf fehl, werden Ersatzwerte verwendet:
- Essen: 967,4 kWh/kWp/Jahr
- Bad Marienberg: 952,5 kWh/kWp/Jahr

In [12]:
def get_pvgis_yield(lat, lon, tilt, azimuth, loss):
    url = "https://re.jrc.ec.europa.eu/api/v5_3/PVcalc"
    params = {
        "lat": lat, "lon": lon, "peakpower": 1,
        "pvtechchoice": "crystSi", "loss": loss,
        "angle": tilt, "aspect": azimuth,
        "outputformat": "json", "raddatabase": "PVGIS-SARAH3",
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        if r.status_code != 200:
            print(f"  WARNUNG: {r.status_code}")
            return None
        val = r.json()["outputs"]["totals"]["fixed"]["E_y"]
        print(f"  -> {val:.1f} kWh/kWp/Jahr  [PVGIS-API]")
        return val
    except Exception as e:
        print(f"  WARNUNG: {e}")
        return None

print(f'Essen ({ESSEN["lat"]}N, {ESSEN["lon"]}E):')
yield_essen = get_pvgis_yield(ESSEN["lat"], ESSEN["lon"], TILT, AZIMUTH, SYSTEM_LOSS)

print(f'Bad Marienberg ({BAD_MARIENBERG["lat"]}N, {BAD_MARIENBERG["lon"]}E):')
yield_bm = get_pvgis_yield(BAD_MARIENBERG["lat"], BAD_MARIENBERG["lon"], TILT, AZIMUTH, SYSTEM_LOSS)

if yield_essen is None:
    yield_essen = 967.4
    print(f"  Ersatzwert: {yield_essen}")
if yield_bm is None:
    yield_bm = 952.5
    print(f"  Ersatzwert: {yield_bm}")

print(f"\nEssen          : {yield_essen:.1f} kWh/kWp/Jahr")
print(f"Bad Marienberg : {yield_bm:.1f} kWh/kWp/Jahr")

Essen (51.455N, 7.011E):
  WARNUNG: HTTPSConnectionPool(host='re.jrc.ec.europa.eu', port=443): Max retries exceeded with url: /api/v5_3/PVcalc?lat=51.455&lon=7.011&peakpower=1&pvtechchoice=crystSi&loss=14&angle=35&aspect=-45&outputformat=json&raddatabase=PVGIS-SARAH3 (Caused by NameResolutionError("HTTPSConnection(host='re.jrc.ec.europa.eu', port=443): Failed to resolve 're.jrc.ec.europa.eu' ([Errno 8] nodename nor servname provided, or not known)"))
Bad Marienberg (50.651N, 7.952E):
  WARNUNG: HTTPSConnectionPool(host='re.jrc.ec.europa.eu', port=443): Max retries exceeded with url: /api/v5_3/PVcalc?lat=50.651&lon=7.952&peakpower=1&pvtechchoice=crystSi&loss=14&angle=35&aspect=-45&outputformat=json&raddatabase=PVGIS-SARAH3 (Caused by NameResolutionError("HTTPSConnection(host='re.jrc.ec.europa.eu', port=443): Failed to resolve 're.jrc.ec.europa.eu' ([Errno 8] nodename nor servname provided, or not known)"))
  Ersatzwert: 967.4
  Ersatzwert: 952.5

Essen          : 967.4 kWh/kWp/Jahr
Bad 

## 9. Zonen-Zuordnung

Jedes Gebaeude wird seiner Referenzstadt zugeordnet — raeumlicher Join gegen
die TRY-Zonenpolygone (`TRY_polygons.geojson`), projiziert in EPSG:25832.

Das legt fest, welchen PVGIS-Ertragswert das Gebaeude erhaelt:
- Zone Essen → 967,4 kWh/kWp/Jahr
- Zone Bad Marienberg → 952,5 kWh/kWp/Jahr

In [13]:
try_gdf = gpd.read_file(TRY_FILE, columns=["geometry", "Station"])
try_gdf.set_crs("EPSG:4236", allow_override=True, inplace=True)
try_gdf.to_crs("EPSG:25832", inplace=True)

gdf = gpd.sjoin(gdf, try_gdf, how="left", predicate="intersects", rsuffix="_try")

gdf.rename(columns={"Station": "nearest_city"}, inplace=True)

n_e  = (gdf["nearest_city"] == "Essen").sum()
n_bm = (gdf["nearest_city"] == "Bad Marienberg").sum()
print(f'Essen          : {n_e:,}  ({n_e/len(gdf)*100:.1f}%)  @ {yield_essen:.1f} kWh/kWp/Jahr')
print(f'Bad Marienberg : {n_bm:,}  ({n_bm/len(gdf)*100:.1f}%)  @ {yield_bm:.1f} kWh/kWp/Jahr')

Essen          : 2,982,956  (72.2%)  @ 967.4 kWh/kWp/Jahr
Bad Marienberg : 1,128,219  (27.3%)  @ 952.5 kWh/kWp/Jahr


## 10. Waermebedarf (TABULA × Al)

Der Waermebedarf wird direkt aus den TABULA-Kennwerten abgeleitet:

```
heat_total_kwh = (heat_EUI + DHW_EUI) × Al_m2
```

mit:
- `heat_EUI` — spezifischer Waermebedarf aus TABULA je TABULA-Typ und
  Sanierungszustand
- `DHW_EUI` — Warmwasserbedarf aus TABULA
- `Al_m2` — konditionierte Wohnflaeche aus Abschnitt 6

Jedes Gebaeude erhaelt seinen eigenen Waermewert anhand des tatsaechlich
zugeordneten Sanierungszustands (Zustand 1 = Original, Zustand 2 = uebliche
Sanierung, Zustand 3 = fortschrittliche Sanierung).

In [14]:
heat_raw  = pd.read_excel(HEAT_FILE, sheet_name="Germany")

heat_raw.rename(columns={
    "existing state (Heating)": "spec_heat_1",
    "Usual refurbishment":      "spec_heat_2",
    "advanced refurbishment":   "spec_heat_3",
    "tabula_type": "tabula_key"
}, inplace=True)

heat_raw

,tabula_key,spec_heat_1,domestic hot water system,spec_heat_2,spec_heat_3
0,DE.N.AB.02,111.9,15,72.1,15.5
1,DE.N.AB.03,129.2,15,71.0,21.4
2,DE.N.AB.04,127.7,15,70.0,20.7
3,DE.N.AB.05,116.9,15,65.3,18.1
4,DE.N.AB.06,94.5,15,58.1,14.3
5,DE.N.AB.07,94.5,15,58.1,14.3
6,DE.N.AB.08,94.5,15,58.1,14.3
7,DE.N.AB.09,94.5,15,58.1,14.3
8,DE.N.AB.10,94.5,15,58.1,14.3
9,DE.N.AB.11,94.5,15,58.1,14.3


In [15]:
gdf["tabula_key"]              = gdf["tabula_type"].str.split(".").str[:4].str.join(".")
gdf["refurbishment_state_str"] = gdf["refurbishment_state"].astype(str)

gdf = gdf.merge(
    heat_raw,
    left_on="tabula_key",
    right_on="tabula_key",
    how="left",
    suffixes=("", "_heat")
)

In [16]:
# Warmwasser zum Start nicht einbeziehen
gdf["heat_demand_1"] = (gdf["spec_heat_1"]) * gdf["Al_m2"]
gdf["heat_demand_2"] = (gdf["spec_heat_2"]) * gdf["Al_m2"]
gdf["heat_demand_3"] = (gdf["spec_heat_3"]) * gdf["Al_m2"]
gdf["heat_total_kwh"] = np.select(
    [
        gdf["refurbishment_state_str"] == "1",
        gdf["refurbishment_state_str"] == "2",
        gdf["refurbishment_state_str"] == "3",
    ],
    [
        gdf["spec_heat_1"] * gdf["Al_m2"],
        gdf["spec_heat_2"] * gdf["Al_m2"],
        gdf["spec_heat_3"] * gdf["Al_m2"],
    ],
    default=0
)

gdf["dhw"] = gdf["domestic hot water system"] * gdf["Al_m2"]

print(f'Zugeordnet : {gdf["heat_demand_1"].notna().sum():,} Gebaeude')
print(f'Fehlend    : {gdf["heat_demand_1"].isna().sum():,} Gebaeude')
print(f'Waerme gesamt: {gdf["heat_demand_1"].sum()/1e9:.2f} TWh/Jahr')
print(f'Waerme mittel : {gdf["heat_demand_1"].mean():,.0f} kWh/Jahr')
print(f"\nHinweis: Waerme ist ~4x Strom — fuer Deutschland erwartungsgemaess")

Zugeordnet : 4,134,282 Gebaeude
Fehlend    : 0 Gebaeude
Waerme gesamt: 125.36 TWh/Jahr
Waerme mittel : 30,323 kWh/Jahr

Hinweis: Waerme ist ~4x Strom — fuer Deutschland erwartungsgemaess


## 11. Parquet speichern

Speichert die gemeinsame Arbeitsdatei `nrw_buildings.parquet` — das Pendant zu
`nl_buildings.parquet` im NL-Repositorium. Alle weiteren Notebooks (02-07)
lesen und reichern genau diese Datei an. Nur die fuer das Clustering
benoetigten Spalten bleiben erhalten.

In [17]:
keep = ["footprint", "id", "height_m", "size_class", "tabula_type",
        "refurbishment_state", "construction_year",
        "area_m2", "heated_height", "Al_m2",
        "tabula_key", "elec_kwh",
        "heat_demand_1", "heat_demand_2", "heat_demand_3", "heat_total_kwh", "dhw",
        "nearest_city"]

out = gdf[[c for c in keep if c in gdf.columns]]
if "footprint" in out.columns:
    out = gpd.GeoDataFrame(out, geometry="footprint", crs=gdf.crs)
out.to_parquet(OUTPUT_FILE, index=False)

print(f"Gespeichert: {len(out):,} Gebaeude -> {OUTPUT_FILE}")
print("\nFertig.")

Gespeichert: 4,134,282 Gebaeude -> ./nrw_buildings.parquet

Fertig.
